In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
df = pd.read_csv("../data/features/features.csv")

df = df.sort_values(["patient_id", "hour_from_admission"]).reset_index(drop=True)

df.head()

,hour_from_admission,heart_rate,respiratory_rate,spo2_pct,temperature_c,systolic_bp,diastolic_bp,oxygen_device,oxygen_flow,mobility_score,...,lactate_roll_mean,lactate_roll_std,lactate_cv,tachycardia_flag,temp_diff,fever_trend,hour_sin,hour_cos,time_bucket,sofa_proxy
0,0,68.58,14.47,96.52,37.18,108.94,78.43,4,0.0,2,...,1.183333,0.063456,0.053625,0,0.07,0,0.000000,1.000000,0,-2.91
1,1,67.03,13.87,94.94,37.25,111.73,79.14,4,0.0,3,...,1.183333,0.063456,0.053625,0,0.07,0,0.258819,0.965926,0,-4.27
2,2,69.05,14.63,94.45,37.29,111.48,78.86,4,0.0,2,...,1.183333,0.063456,0.053625,0,0.04,0,0.500000,0.866025,0,-3.47
3,3,69.07,14.42,95.16,37.27,110.68,76.79,4,0.0,2,...,1.183333,0.063456,0.053625,0,-0.02,0,0.707107,0.707107,0,-3.47
4,4,73.35,15.62,95.83,37.21,110.38,75.47,4,0.0,3,...,1.183333,0.063456,0.053625,0,-0.06,0,0.866025,0.500000,0,-3.80


In [3]:
TARGET = "deterioration_next_12h"

y = df[TARGET]
groups = df["patient_id"]

In [4]:
patient_df = df.groupby("patient_id")[TARGET].max().reset_index()

patient_ids = patient_df["patient_id"]
patient_labels = patient_df[TARGET]

In [5]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

folds = []

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(patient_ids, patient_labels, patient_ids)
):
    train_patients = patient_ids.iloc[train_idx]
    val_patients = patient_ids.iloc[val_idx]

    folds.append((train_patients, val_patients))

In [6]:
import os

os.makedirs("../data/splits", exist_ok=True)

for i, (train_p, val_p) in enumerate(folds):
    
    train_df = df[df["patient_id"].isin(train_p)]
    val_df = df[df["patient_id"].isin(val_p)]
    
    train_df.to_csv(f"../data/splits/train_fold_{i}.csv", index=False)
    val_df.to_csv(f"../data/splits/val_fold_{i}.csv", index=False)

print("Saved 5-fold splits.")

Saved 5-fold splits.


In [7]:
for i, (train_p, val_p) in enumerate(folds):
    overlap = set(train_p).intersection(set(val_p))
    print(f"Fold {i} overlap:", len(overlap))

Fold 0 overlap: 0
Fold 1 overlap: 0
Fold 2 overlap: 0
Fold 3 overlap: 0
Fold 4 overlap: 0


In [8]:
for i in range(5):
    train_df = pd.read_csv(f"../data/splits/train_fold_{i}.csv")
    val_df = pd.read_csv(f"../data/splits/val_fold_{i}.csv")
    
    print(f"\nFold {i}")
    print("Train ratio:\n", train_df[TARGET].value_counts(normalize=True))
    print("Val ratio:\n", val_df[TARGET].value_counts(normalize=True))


Fold 0
Train ratio:
 deterioration_next_12h
0    0.946121
1    0.053879
Name: proportion, dtype: float64
Val ratio:
 deterioration_next_12h
0    0.945269
1    0.054731
Name: proportion, dtype: float64

Fold 1
Train ratio:
 deterioration_next_12h
0    0.945232
1    0.054768
Name: proportion, dtype: float64
Val ratio:
 deterioration_next_12h
0    0.948785
1    0.051215
Name: proportion, dtype: float64

Fold 2
Train ratio:
 deterioration_next_12h
0    0.9467
1    0.0533
Name: proportion, dtype: float64
Val ratio:
 deterioration_next_12h
0    0.942985
1    0.057015
Name: proportion, dtype: float64

Fold 3
Train ratio:
 deterioration_next_12h
0    0.94474
1    0.05526
Name: proportion, dtype: float64
Val ratio:
 deterioration_next_12h
0    0.950866
1    0.049134
Name: proportion, dtype: float64

Fold 4
Train ratio:
 deterioration_next_12h
0    0.94696
1    0.05304
Name: proportion, dtype: float64
Val ratio:
 deterioration_next_12h
0    0.941875
1    0.058125
Name: proportion, dtype: float6

In [9]:
cutoff = df["hour_from_admission"].quantile(0.8)

train_temp = df[df["hour_from_admission"] <= cutoff]
test_temp = df[df["hour_from_admission"] > cutoff]

train_temp.to_csv("../data/splits/train_temporal.csv", index=False)
test_temp.to_csv("../data/splits/test_temporal.csv", index=False)

print("Temporal split created.")

Temporal split created.


In [10]:
patient_df.to_csv("../data/splits/patient_labels.csv", index=False)

In [11]:
print("Total patients:", df["patient_id"].nunique())
print("Total rows:", len(df))

Total patients: 7000
Total rows: 293248
